<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_11_3_mcp_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 11: Model Context Protocol (MCP)**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 11 Material

* Part 11.1: Introduction to the Model Context Protocol [[Notebook]](t81_559_class_11_1_mcp.ipynb)
* Part 11.2: Using MCP Servers from an Agent [[Notebook]](t81_559_class_11_2_mcp_client.ipynb)
* **Part 11.3: Building Your Own MCP Server** [[Notebook]](t81_559_class_11_3_mcp_server.ipynb)
* Part 11.4: MCP Resources and Multi-Server Agents [[Notebook]](t81_559_class_11_4_mcp_multi.ipynb)
* Part 11.5: MCP Security and the Road Ahead [[Notebook]](t81_559_class_11_5_mcp_security.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [1]:
import sys

# MCP launches its servers as subprocesses and hands them this notebook's
# stderr, which must have a real file descriptor. Notebook kernels do not
# provide one, so we point stderr at a log file. The MCP library captures
# stderr the moment it first loads, so this redirect must run before any
# MCP import, which is why it sits at the top of the setup cell.
sys.stderr = open("mcp_server.log", "w")

import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai langchain-mcp-adapters "mcp<2"

Note: using Google CoLab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


# Part 11.3: Building Your Own MCP Server

Consuming servers is half the story; the other half is packaging your own capabilities so that *any* MCP host can use them. The standard way to do this in Python is **FastMCP**, which ships inside the official `mcp` SDK. FastMCP works like Flask for MCP: you write a plain Python function, add a decorator, and the framework derives the tool's schema from your type hints and docstring, handles the protocol, and runs the transport.

Notice what "derives the schema" means: the typed parameters and descriptions you learned to design in Module 5 are generated for you from ordinary Python annotations. A good docstring is now literally an API contract that a language model will read.

Our first server wraps the safe calculator from Module 7.2 -- the tool we built because language models are unreliable at arithmetic. The `%%writefile` magic saves the server as a normal Python file, exactly as we did for Streamlit apps in Module 10.

In [2]:
%%writefile math_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def safe_calculator(expression: str) -> str:
    """Evaluate a basic math expression, such as '8273 * 1821'.
    Supports +, -, *, /, parentheses, and numbers only."""
    try:
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing math_server.py


That is a complete MCP server: about a dozen lines. `FastMCP("Math")` names it, `@mcp.tool()` publishes the function, and `mcp.run(transport="stdio")` serves it over standard input/output. The file has no LangChain in it and no OpenAI in it -- it could be dropped into Claude Desktop's or Cursor's configuration unchanged.

We now connect to it the same way we connected to the third-party servers in Part 11.2. The client launches `python math_server.py` as a subprocess and performs the discovery handshake.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    },
})

tools = await client.get_tools()
for t in tools:
    print(f"{t.name}: {t.description}")

safe_calculator: Evaluate a basic math expression, such as '8273 * 1821'.
    Supports +, -, *, /, parentheses, and numbers only.


The description you wrote as a docstring came back over the wire as the tool's published contract. Now the payoff: in Module 7.2, we asked the bare model for 8273 * 1821 and it confidently gave a wrong answer. The agent below gets the right one, through your server.

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(
        model=MODEL,
        use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
    )

agent = create_agent(llm, tools)

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is 8273 * 1821?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is 8273 * 1821?
================================== Ai Message ==================================

[{'arguments': '{"expression":"8273 * 1821"}', 'call_id': 'call_MFQdF9eImKUqkIdUDOC0BlJs', 'name': 'safe_calculator', 'type': 'function_call', 'id': 'fc_08f10ca20d205f44006a6e29fdbcf0819692155d17ef65c86c', 'status': 'completed'}]
Tool Calls:
  safe_calculator (call_MFQdF9eImKUqkIdUDOC0BlJs)
 Call ID: call_MFQdF9eImKUqkIdUDOC0BlJs
  Args:
    expression: 8273 * 1821
================================= Tool Message =================================
Name: safe_calculator

[{'type': 'text', 'text': '15065133', 'id': 'lc_b3ecf5e1-cce4-4ccd-b948-b5d306abc887'}]
================================== Ai Message ==================================

[{'type': 'text', 'text': '15,065,133', 'annotations': [], 'id': 'msg_08f10ca20d205f44006a6e2a02177c81969aef7dc742710cf8', 'phase': 'final_answer'}]


## A Stateful Server: The Toy Car

Servers are long-running processes, so they can hold state between tool calls. To see this, we rebuild Module 7.5's toy car -- three buttons that move a little vehicle around a grid -- as an MCP server. The car's position lives *in the server*; each button press updates it and reports what happened.

In [5]:
%%writefile car_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Car")

x, y = 0, 0
direction = "north"
DIRECTIONS = ["north", "east", "south", "west"]

@mcp.tool()
def press_button(button_color: str) -> str:
    """Press one button on the car's control panel.
    The red button moves the car forward 1 unit.
    The green button turns the car 90 degrees left.
    The yellow button turns the car 90 degrees right."""
    global x, y, direction
    color = button_color.strip().lower()
    if color == "red":
        if direction == "north": y += 1
        elif direction == "east": x += 1
        elif direction == "south": y -= 1
        elif direction == "west": x -= 1
        result = "The button glows red, you move forward one unit."
    elif color == "green":
        direction = DIRECTIONS[(DIRECTIONS.index(direction) - 1) % 4]
        result = "The button glows green, you turn 90 degrees to the left."
    elif color == "yellow":
        direction = DIRECTIONS[(DIRECTIONS.index(direction) + 1) % 4]
        result = "The button glows yellow, you turn 90 degrees to the right."
    else:
        result = "The button buzzes, error."
    return f"{result} Position: ({x},{y}) facing {direction}."

@mcp.tool()
def get_position() -> str:
    """Report the car's current position and heading."""
    return f"The car is at ({x},{y}) facing {direction}."

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing car_server.py


We ask the agent to drive the car in a rectangle, exactly as in Module 7.5 -- but this time the "car" is a separate process that any MCP host could drive.

In [6]:
client = MultiServerMCPClient({
    "car": {
        "transport": "stdio",
        "command": "python",
        "args": ["car_server.py"],
    },
})

tools = await client.get_tools()
agent = create_agent(llm, tools)

async for step in agent.astream(
    {"messages": [{"role": "user", "content":
        "Push the buttons in a way that causes the car to move in a rectangle, "
        "then confirm the final position."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Push the buttons in a way that causes the car to move in a rectangle, then confirm the final position.
================================== Ai Message ==================================

[{'id': 'rs_03fdcd29e501093f006a6e2a06263481979ec30e3856850575', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqbioG8ZG3ESiI4GMcUtKMowrb5MIn1FVyD513XfoAP7OJDSnfqioXcnxaJyHf7v86aH-Gnkq4Cph5TyKUVwUe1ubjNvNc-FejPfsJO5hy3z7g0RcX5OgZF76qPLku_IT8ZyIXxGQIQ6wOH_99PCygFZxBQyc4tkDfqjJXSrqrrj2zISLh4mYDG6XGh1veRWt1NvcuDO94zF7VaZjgyrwy2yMaFlo9WWZztcpKUaFBQrvWLKJZen-GFMawIc060v4s2qpra6qH8J_cGyUXiZMxq8kwriftEZdPprme47gTlEvVDTIGVkUQP3lvxdWGcHXtWcYLUT5HwsL6TWo4DvlJt9ejT2kq-TfHyxsBxNBaPzzTG0FzpniOk8uunv-vU0_vCTOYugpi_rcV3_-HxifhF_yCd2SN8PijspD7tzVU_ycBCzPLMPyARIKtySUfI9hxiHkY2CUpJnGqh7IESrS4MWU8zdxn38wKbmeaF8wWTPUOO8bFZVYLdD2DD6ZN5LghxfhUW25UPutA08MmALo_Pghob2_IOX4jemPaKEJfB7PVGyEhTIJx6T4KwURl6v-85WfileCvcrq

Note one subtlety: the client launches a *fresh* server process for its connection, so state persists across tool calls within a conversation, but restarting the client resets the car. Production servers persist state to files or databases for exactly this reason.

You now know both sides of the protocol. In the next part we go beyond tools -- servers that publish *data* as resources -- and connect one agent to several servers at once.